# CIRSOC 301-2018 — Verificación Sección Doble T

**Autor:** Aníbal Mieres — anibalmieres@gmail.com  
**Versión:** 1.0.0  
© 2026 Aníbal Mieres. Todos los derechos reservados.  
*Este documento no puede ser utilizado en memorias de cálculo o trabajos técnicos de terceros sin autorización escrita del autor.*

---

Verificación de resistencia de diseño para secciones doble T simétricas  
(W, HEA, HEB, IPE, IPN, HP, S, M) según **CIRSOC 301-2018**.

| Bloque | Ref. | Descripción |
|--------|------|-------------|
| Clasificación | B.4-1 | Compacta / No compacta / Esbelta |
| E.3 | §E.3 | Compresión — pandeo flexional |
| E.4 | §E.4 | Compresión — pandeo torsional |
| F.2 | §F.2 | Flexión eje fuerte — plastificación + PLT |
| F.6 | §F.6 | Flexión eje débil |
| G.2 | §G.2 | Corte eje fuerte (alma) |
| G.6 | §G.6 | Corte eje débil (alas) |
| D.2 | §D.2/D.3/J.4.3 | Tracción — fluencia / rotura / bloque de corte |
| H.1 + G.7 | §H.1/§G.7 | Combinaciones por caso de carga |

## 0. Configuración

In [ ]:
# =============================================================================
# Copyright (c) 2026 Aníbal Mieres — anibalmieres@gmail.com
# Todos los derechos reservados.
# =============================================================================
import sys, pathlib

ROOT = pathlib.Path.cwd().parent   # raíz del repositorio cirsoc301/
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from calc.perfiles     import cargar_doblet, listar_familias, filtrar_familia
from calc.doble_t      import (Material, InputsEstructurales, InputsUnion,
                                CasoCarga, verificar_doblet)
from calc.trazabilidad import mostrar_bloque, mostrar_resumen, mostrar_memoria

print("Módulos cargados correctamente.")

## 1. Base de datos de perfiles

Los perfiles se cargan desde `data/perfiles_doblet.csv`.  
Para **agregar o corregir un perfil**: editá el CSV con Excel y volvé a ejecutar esta celda.

In [ ]:
perfiles = cargar_doblet()
print(f"Perfiles cargados: {len(perfiles)}")
for fam in listar_familias(perfiles):
    g = filtrar_familia(perfiles, fam)
    ns = list(g.keys())
    print(f"  {fam:<10} {len(ns):>3} perfiles — {ns[0]} … {ns[-1]}")

In [ ]:
# ── SELECCIONAR PERFIL ────────────────────────────────────────────────
# Familias disponibles: HEA, HEB, W, HP, IPE, IPN, S, M
NOMBRE_PERFIL = 'W14x53'

p = perfiles[NOMBRE_PERFIL]
print(f"Perfil: {p.nombre}  [{p.familia}]  fuente: {p.fuente}")
print()
props = [("Ag",p.Ag,"cm²"),("g",p.g,"kg/m"),("h",p.h,"cm"),("bf",p.bf,"cm"),
         ("tf",p.tf,"cm"),("tw",p.tw,"cm"),("hw",p.hw,"cm"),("b=bf/2",p.b,"cm"),
         ("Ix",p.Ix,"cm⁴"),("Iy",p.Iy,"cm⁴"),("Sx",p.Sx,"cm³"),("Sy",p.Sy,"cm³"),
         ("Zx",p.Zx,"cm³"),("Zy",p.Zy,"cm³"),("rx",p.rx,"cm"),("ry",p.ry,"cm"),
         ("J",p.J,"cm⁴"),("Cw",p.Cw,"cm⁶")]
for i in range(0, len(props), 2):
    a, b_ = props[i], props[i+1]
    print(f"  {a[0]:<8} = {a[1]:>12.4g} {a[2]:<5}    {b_[0]:<8} = {b_[1]:>12.4g} {b_[2]}")

## 2. Material

In [ ]:
# Opciones: 'F24' (Fy=235 MPa) | 'F36' (Fy=355 MPa)
MATERIAL = 'F36'

mat = Material.desde_nombre(MATERIAL)
print(f"Material : {mat.nombre}")
print(f"Fy       : {mat.Fy} MPa")
print(f"Fu       : {mat.Fu} MPa")
print(f"Fr       : {mat.Fr} MPa  (tensión residual §F.2)")

## 3. Parámetros del miembro

Longitudes de pandeo, coeficientes k, distancia entre arriostramientos laterales  
y factor de gradiente de momento Cb.

In [ ]:
ie = InputsEstructurales(
    Lx = 0.0,     # cm  longitud de pandeo eje fuerte
    kx = 1.0,     # —   coeficiente eje fuerte  (1.0=art-art, 0.5=emp-emp, 2.0=ménsula)
    Ly = 0.0,     # cm  longitud de pandeo eje débil
    ky = 1.0,     # —   coeficiente eje débil
    Lz = 0.0,     # cm  longitud de pandeo torsional
    kz = 1.0,
    Lb = 0.0,     # cm  distancia entre arriostramientos laterales
    Cb = 1.0,     # —   factor de gradiente de momento (1.0 = conservador)
    carga_pos = 'alma',   # 'alma' | 'ala_sup'
)
print(ie)

## 4. Tipo de unión (tracción §D.2)

| `tipo`            | Descripción |
|-------------------|-------------|
| `soldada_total`   | Soldadura continua → An=Ag, U=1 |
| `soldada_parcial` | Soldadura parcial → ingresar `An_manual` y `U_manual` |
| `bulones`         | Unión bulonada → ingresar geometría o `An_manual` |

In [ ]:
# ── MODO 1: soldadura total (más común) ──────────────────────────────
union = InputsUnion(tipo='soldada_total')


# ── MODO 2: An manual (descomentar para usar) ───────────────────────────
# union = InputsUnion(
#     tipo      = 'soldada_parcial',
#     An_manual = 0.0,    # cm²  área neta efectiva
#     U_manual  = 0.90,   # —    factor de corte diferido (Tabla D.3)
# )


# ── MODO 3: bulones — cálculo automático de An (descomentar para usar) ─────────
# union = InputsUnion(
#     tipo                 = 'bulones',
#     db                   = 0.0,    # cm  diámetro nominal del bulón
#     n_bulones_linea      = 0,
#     n_lineas             = 1,
#     elementos_conectados = 'ala',  # 'ala' | 'alma' | 'todos'
#     t_elemento           = None,
#     Agv = 0.0,
#     Agt = None,
# )

print(f"Tipo de unión: {union.tipo}")

## 5. Casos de carga

Ingresar las solicitaciones de diseño para cada caso.  
**Convención:** Pu(+) = tracción · Pu(-) = compresión  
Eje fuerte = y-y → Muy, Vuz · Eje débil = z-z → Muz, Vuy

> **Nota sobre integración futura:** Se prevé importar estas solicitaciones
> directamente desde archivos de resultados de STAAD.Pro (.ANL),
> Autodesk Robot (.RTD/.CSV) u OpenSees. Por ahora se ingresan manualmente.

In [ ]:
# Completar con las solicitaciones de diseño.
# Agregar o eliminar bloques CasoCarga según necesidad.

casos = [
    CasoCarga(
        label = 'Caso 1',
        desc  = '',          # descripción libre (ej: 'Máx. compresión axial')
        Pu    = 0.0,         # kN   axial (+ tracción, - compresión)
        Muy   = 0.0,         # kNm  flexión eje fuerte
        Muz   = 0.0,         # kNm  flexión eje débil
        Vuz   = 0.0,         # kN   corte eje fuerte
        Vuy   = 0.0,         # kN   corte eje débil
    ),
    CasoCarga(
        label = 'Caso 2',
        desc  = '',
        Pu    = 0.0,
        Muy   = 0.0,
        Muz   = 0.0,
        Vuz   = 0.0,
        Vuy   = 0.0,
    ),
    # CasoCarga(
    #     label = 'Caso 3',
    #     desc  = '',
    #     Pu    = 0.0, Muy = 0.0, Muz = 0.0, Vuz = 0.0, Vuy = 0.0,
    # ),
]

print(f"{len(casos)} caso(s) de carga definido(s).")

## 6. Cálculo

In [ ]:
resultado = verificar_doblet(
    perfil   = p,
    material = mat,
    ie       = ie,
    union    = union,
    casos    = casos,
)
print("Verificación ejecutada correctamente.")

## 7. Resumen ejecutivo

In [ ]:
mostrar_resumen(resultado)

## 8. Memoria de cálculo completa

Cada bloque muestra la expresión simbólica con valores sustituidos,  
el resultado numérico y la referencia a la cláusula CIRSOC 301-2018.

In [ ]:
mostrar_memoria(resultado)

## 9. Validación cruzada con planilla de referencia

Comparación contra la planilla original:  
`MC_em - TIPICO - verificaciones_CIRSOC_301_2018 - 18-10-2024.xlsx`, pestaña *Doble T*.  
Perfil de referencia: **W14×53 / F36**. Tolerancia aceptada: < 0.05%.

In [ ]:
p_ref    = perfiles['W14x53']
mat_ref  = Material.desde_nombre('F36')
ie_ref   = InputsEstructurales(Lx=388.5,kx=2.5,Ly=480,ky=1,Lz=480,kz=1,Lb=480,Cb=1)
casos_ref = [
    CasoCarga('Caso 1','Máx. flexión eje fuerte', Pu=-309.48,Muy=17.59,Muz=11.21,Vuz=2.20, Vuy=15.54),
    CasoCarga('Caso 2','Máx. corte eje fuerte',   Pu=-126.43,Muy=11.88,Muz=0.90, Vuz=12.13,Vuy=1.58),
    CasoCarga('Caso 3','Máx. compresión axial',   Pu=-614.68,Muy=0.01, Muz=7.10, Vuz=0.41, Vuy=6.59),
    CasoCarga('Caso 4','Máx. tracción axial',     Pu=272.01, Muy=0.00, Muz=5.50, Vuz=0.00, Vuy=4.29),
]
res_ref = verificar_doblet(p_ref, mat_ref, ie_ref, InputsUnion(), casos_ref)
r_ref   = res_ref.resistencias()

REF_RES  = {'Pd_comp':1465.43,'Pdt':3214.17,'Mdx':333.73,'Mdy':112.14,'Vdz':637.90,'Vdy':660.21}
REF_CASO = {'Caso 1':0.3469,'Caso 2':0.0868,'Caso 3':0.4758,'Caso 4':0.0914}
TOL      = 0.05

print(f"{\"Símbolo\":<12}{\"Calculado\":>12}{\"Referencia\":>12}{\"Dif%\":>8}  Estado")
print("─"*50)
all_ok = True
for sym, ref in REF_RES.items():
    calc = r_ref[sym][0]; dif = abs(calc-ref)/ref*100; ok = dif < TOL; all_ok = all_ok and ok
    print(f"  {sym:<10}{calc:>12.2f}{ref:>12.2f}{dif:>7.3f}%  {'\u2713' if ok else '\u2717 REVISAR'}")
print()
for c in res_ref.casos:
    ref = REF_CASO.get(c['label'])
    if ref is None: continue
    dif = abs(c['ratio']-ref)/ref*100; ok = dif < TOL; all_ok = all_ok and ok
    print(f"  {c['label']:<10}{c['ratio']:>12.4f}{ref:>12.4f}{dif:>7.3f}%  {'\u2713' if ok else '\u2717 REVISAR'}")
print()
print("═"*50)
print(f"  {'\u2713 TODOS LOS VALORES COINCIDEN' if all_ok else '\u2717 HAY DIFERENCIAS — REVISAR'}")
print("═"*50)

---
## Cómo usar este notebook

**Cambiar perfil:** Modificar `NOMBRE_PERFIL` en la celda 1 y ejecutar *Run All*.

**Agregar un perfil al CSV:**  
Abrir `data/perfiles_doblet.csv`, agregar una fila y completar la columna `fuente`.

**Agregar un caso de carga:**  
Copiar uno de los bloques `CasoCarga(...)` en la celda 5 y completar los valores.

**Cambiar tipo de unión:**  
Descomentar el modo deseado en la celda 4.

---
© 2026 Aníbal Mieres — anibalmieres@gmail.com — Todos los derechos reservados